In [ ]:
import pandas as pd

ids = ['1844586908208', '3213986908117', '4825086908855']

# Tab 1: full sel_col row per policy (3 rows)
raw = (ltv
       .filter(F.col('ply_policy_id').isin(ids))
       .select(sel_col)
       .toPandas()
       .set_index('ply_policy_id')
       .loc[ids]                      # preserve show(3) order
       .reset_index())

# Tab 2: stored vs recalc + ratio, per policy per item (tidy/long)
yd = (1 + 0.03) / (1 + 0.072)
disc = yd**0.25 + yd**0.75            # the 0.25/0.75 term, ~1.9601

rows = []
for _, r in raw.iterrows():
    for item in ['lifetime', 'overhead', 'claims', 'commissions', 'acquisition', 'marketing']:
        # new-business premium base differs for acq/mkt
        if item in ('acquisition', 'marketing'):
            base_new = r['drv_full_premium_amt'] * disc     # PARKED: annual base, pending Yinan
        else:
            base_new = r['premium_new']

        recalc_new   = base_new        * r[f'expense_ratio_e006scl_{item}_new']
        recalc_renew = r['premium_renew'] * r[f'expense_ratio_e006scl_{item}_renew']
        stored_new   = r[f'e006scl_{item}_exp_new']
        stored_renew = r[f'e006scl_{item}_exp_renew']

        rows.append({
            'ply_policy_id': r['ply_policy_id'],
            'item': item,
            'stored_new': stored_new, 'recalc_new': recalc_new,
            'ratio_new': stored_new / recalc_new if recalc_new else None,
            'stored_renew': stored_renew, 'recalc_renew': recalc_renew,
            'ratio_renew': stored_renew / recalc_renew if recalc_renew else None,
        })

check = pd.DataFrame(rows)
check['ratio_ok'] = check['ratio_new'].round(4).isin([1.0]) | check['ratio_new'].isna()

# Export both tabs
out = '/home/<you>/expense_validation_3policies.xlsx'   # <-- set your path
with pd.ExcelWriter(out, engine='openpyxl') as xw:
    raw.to_excel(xw, sheet_name='raw', index=False)
    check.to_excel(xw, sheet_name='recalc_check', index=False)
print('wrote', out)